# Recursos da interface (E2, caracterização)

Conta qubits, portas de dois qubits e profundidade das peças da interface e ajusta como esses recursos crescem com o número de arcos m:

- **preparação A_k** (Dicke de Bärtschi–Eidenbenz, uniforme sobre F_k);
- **oráculo de tabela-verdade** (marca as M interdições ótimas; usado na demonstração em hardware);
- **oráculo aritmético sobre (x, α)**: γ_a = [α_i = 0 ∧ α_j = 1 ∧ x_a = 0] por portas multicontroladas, soma ponderada das capacidades (`WeightedSumGate`), comparação com τ (`IntegerComparatorGate`), fase e descomputação. Validado em `tests/test_oracle.py` em todos os estados de base de uma instância minúscula.

Duas contagens: **lógica** (base `cx`, `u`, sem restrição de conectividade) e **transpilada** para o `FakeFez` (Heron, 156 qubits, nível de otimização 1), esta só para circuitos de até 156 qubits.

Instâncias (protocolo, desvio 8.4):
- família aleatória de E2 (`arc_factor` 2, custo unitário, k ∈ {1, 2} com k < κ, semente 0), n de 3 a 40. As nove instâncias com n de 3 a 7 são as do experimento de 06/11 e servem de **controle**: as contagens precisam sair idênticas às gravadas;
- semente 0 de cada combinação (w, k) de E1, lidas da pasta de E1. Nelas o oráculo de tabela-verdade não é contado, porque marcar os ótimos exigiria enumerar até 974.121 planos.

Colunas analíticas: a versão (x, α) busca sobre |F_k| · 2^(n−2) pares, com O(√(|F_k| · 2^(n−2))) consultas; ela só supera a enumeração clássica de F_k (|F_k| avaliações) quando 2^(n−2) < |F_k|.

## 1. Imports, instâncias e constantes

In [ ]:
import math

import matplotlib.pyplot as plt
import pandas as pd
from qiskit_ibm_runtime.fake_provider import FakeFez

from qmfnip.evaluation import fit_scaling_exponent
from qmfnip.grover import search_space
from qmfnip.instances import Scenario, full_cut_cost, max_flow
from qmfnip.oracle import arithmetic_oracle, circuit_resources, phase_oracle, state_preparation
from qmfnip.outputs import ExperimentOutput, capture_environment, latest_experiment

In [ ]:
SCENARIO = Scenario(
    "random",
    sizes=(3, 4, 5, 6, 7, 8, 10, 12, 16, 20, 24, 32, 40),
    budgets=(1, 2),
    seeds=(0,),
    params={"arc_factor": 2.0, "cost_range": (1, 1)},
    skip_degenerate=True,
)
CONTROL_SIZES = (3, 4, 5, 6, 7)
E1_TAG = "durr_hoyer-random_order-enumeration__layered__n6-27__runs200"
E2_ORIGINAL_TAG = "dicke-truth_table-arithmetic__random__n3-7__runs1"
E1_SEED = 0
BACKEND = FakeFez()
SEED = 0
N_BOOT = 2000
COMPARED = ["logical_qubits", "physical_qubits", "two_qubit_gates", "total_gates", "circuit_depth", "two_qubit_depth"]

Lista de instâncias. Nas aleatórias, o limiar τ = F\* + 1 e os planos marcados saem da enumeração de F_k. Nas de E1, a instância é lida da pasta do experimento (o hash é conferido) e F\* vem de `t1_instances.csv`, certificado pelo MILP.

In [ ]:
E1 = latest_experiment(contains=E1_TAG)
e1_table = pd.read_csv(E1.path / "t1_instances.csv").set_index("instance_id")
e1_instances = E1.load_instances()
cases = []
for instance in SCENARIO.instances():
    space = search_space(instance)
    cases.append(
        {
            "family": "random", "instance": instance, "N": space.size, "M": space.optimal_count,
            "optimal_flow": space.optimum, "marked": space.interdictions[space.optimal_indices].astype(int).tolist(),
        }
    )
for name, row in e1_table[e1_table["instance_seed"] == E1_SEED].sort_values(["n_nodes", "k"]).iterrows():
    instance = e1_instances[name]
    assert instance.budget == row["k"] and instance.m == row["n_arcs"]
    cases.append(
        {
            "family": "layered", "instance": instance, "N": int(row["N"]), "M": int(row["M"]),
            "optimal_flow": int(row["optimal_flow"]), "marked": None,
        }
    )
pd.DataFrame(
    [
        {"family": case["family"], "instance_id": case["instance"].instance_id, "n": case["instance"].n,
         "m": case["instance"].m, "k": case["instance"].budget, "N": case["N"], "M": case["M"]}
        for case in cases
    ]
)

## 2. Execução: montagem e contagem dos circuitos

In [ ]:
output = ExperimentOutput.create(
    ["dicke", "truth_table", "arithmetic"], "random-layered", [case["instance"].n for case in cases], 1
)
output.write_json("environment.json", capture_environment())
output.update_manifest(
    {
        "status": "running", "scenario": SCENARIO.to_dict(), "e1_source": E1.path.name, "e1_seed": E1_SEED,
        "backend": BACKEND.name, "seed": SEED, "protocol": "docs/protocol.md, deviation 8.4",
    }
)
rows = []
for case in cases:
    instance = case["instance"]
    output.write_instance(instance)
    circuits = {
        "dicke_preparation": state_preparation(instance.m, instance.budget),
        "arithmetic_oracle": arithmetic_oracle(instance, case["optimal_flow"] + 1),
    }
    if case["marked"] is not None:
        circuits["truth_table_oracle"] = phase_oracle(instance.m, case["marked"])
    registers = {f"register_{register.name}": register.size for register in circuits["arithmetic_oracle"].qregs}
    for name, circuit in circuits.items():
        for backend in (None, BACKEND):
            if backend is not None and circuit.num_qubits > backend.num_qubits:
                continue
            resources = circuit_resources(circuit, backend, seed=SEED)
            resources.pop("operations")
            rows.append(
                {
                    "family": case["family"], "instance_id": instance.instance_id, "n": instance.n, "m": instance.m,
                    "k": instance.budget, "kappa": full_cut_cost(instance), "N": case["N"], "M": case["M"],
                    "flow_without_interdiction": max_flow(instance), "optimal_flow": case["optimal_flow"],
                    "circuit": name, **resources, **(registers if name == "arithmetic_oracle" else {}),
                }
            )
resources = pd.DataFrame(rows)
output.write_table("resources", resources)
output.update_manifest({"status": "completed"})
resources.groupby(["family", "circuit", "backend"]).size()

## 3. Avaliação: controle, tabela, ajuste de crescimento e figura

Controle: as nove instâncias com n de 3 a 7 precisam reproduzir as contagens gravadas em 06/11 (todas as colunas, menos o tempo de compilação).

In [ ]:
original = pd.read_csv(latest_experiment(contains=E2_ORIGINAL_TAG).path / "resources.csv")
keys = ["instance_id", "circuit", "backend"]
control = resources[(resources["family"] == "random") & resources["n"].isin(CONTROL_SIZES)]
merged = original[keys + COMPARED].merge(control[keys + COMPARED], on=keys, how="outer", suffixes=("_stored", "_new"), indicator=True)
assert (merged["_merge"] == "both").all(), merged[merged["_merge"] != "both"]
differences = {column: int((merged[f"{column}_stored"] != merged[f"{column}_new"]).sum()) for column in COMPARED}
assert not any(differences.values()), differences
control_result = {"rows": len(merged), "instances": merged["instance_id"].nunique(), "differences": differences}
output.write_json("control.json", control_result)
control_result

In [ ]:
wide = resources.pivot_table(
    index=["family", "instance_id", "n", "m", "k", "N", "M"],
    columns=["circuit", "backend"],
    values=["logical_qubits", "two_qubit_gates", "circuit_depth"],
    aggfunc="first",
)
wide = wide.sort_index(level=["family", "m", "k"])
output.write_table("t3_resources_wide", wide.reset_index())
analytic = resources.drop_duplicates("instance_id")[["family", "instance_id", "n", "m", "k", "N", "M"]].copy()
analytic["search_qubits_x"] = analytic["m"]
analytic["search_qubits_x_alpha"] = analytic["m"] + analytic["n"] - 2
analytic["pairs"] = analytic["N"] * 2.0 ** (analytic["n"] - 2)
analytic["queries_x"] = (math.pi / 4) * analytic["N"].pow(0.5)
analytic["queries_x_alpha"] = (math.pi / 4) * analytic["pairs"].pow(0.5)
analytic["classical_evaluations"] = analytic["N"]
analytic["x_alpha_beats_enumeration"] = 2.0 ** (analytic["n"] - 2) < analytic["N"]
arithmetic = resources[(resources["circuit"] == "arithmetic_oracle") & (resources["backend"] == "logical")].set_index("instance_id")
analytic["arithmetic_oracle_qubits"] = analytic["instance_id"].map(arithmetic["logical_qubits"])
analytic["arithmetic_oracle_two_qubit_gates"] = analytic["instance_id"].map(arithmetic["two_qubit_gates"])
analytic = analytic.sort_values(["family", "m", "k"])
output.write_table("t3_analytic", analytic)
analytic

Qubits do oráculo aritmético: a soma dos registradores do circuito (x, α, γ, soma, flag, carry e work) tem de dar o total. x tem m qubits, α tem n − 2, γ tem um qubit por arco que não sai de t nem entra em s, e a soma tem ⌊log₂ Σ u_a⌋ + 1 qubits.

Crescimento: ajuste de potência das portas de dois qubits (contagem lógica) contra m, por circuito, em todas as instâncias; a preparação é ajustada separadamente para cada k, porque a teoria dá O(km).

In [ ]:
layout = arithmetic[[column for column in arithmetic.columns if column.startswith("register_")]]
assert (layout.sum(axis=1) == arithmetic["logical_qubits"]).all()
assert (arithmetic["register_x"] == arithmetic["m"]).all() and (arithmetic["register_alpha"] == arithmetic["n"] - 2).all()
logical = resources[resources["backend"] == "logical"].copy()
logical["series"] = logical["circuit"].where(logical["circuit"] != "dicke_preparation", "dicke_preparation_k" + logical["k"].astype(str))
fits = []
for series, group in logical.groupby("series"):
    if group["m"].nunique() < 3:
        continue
    for y in ("two_qubit_gates", "logical_qubits"):
        fit = fit_scaling_exponent(group.assign(series=series), "m", y, by="series", n_boot=N_BOOT, seed=SEED)
        fits.append(fit.assign(m_min=int(group["m"].min()), m_max=int(group["m"].max())))
fits = pd.concat(fits, ignore_index=True)
output.write_table("growth_fits", fits)
fits.round(3)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for axis, backend in zip(axes, ("logical", BACKEND.name)):
    for index, (name, group) in enumerate(resources[resources["backend"] == backend].groupby("circuit", sort=False)):
        group = group.sort_values(["m", "k"])
        axis.plot(group["m"] + 0.08 * (group["k"] - 1), group["two_qubit_gates"], marker="o", ls="", color=f"C{index}", label=name)
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set_xlabel("arcs m")
    axis.set_title(f"two-qubit gates ({backend})")
    axis.grid(alpha=0.3, which="both", lw=0.4)
axes[0].set_ylabel("two-qubit gates")
axes[1].legend(fontsize=8)
figure.tight_layout()
figure.savefig(output.figure_path("resources_two_qubit_gates"), bbox_inches="tight")